# Photometry data preprocessing

This is a tutorial explaining preprocessing methods for fiber photometry data used in this analysis pipeline.
We will follow these steps:

1. Clean high electrical artifacts with Hampel filtering.

2. Manually indicate the timestamps of big artifacts due to disconnections.

3. Movement correction by subtracting a linear fit of the movement control channel (isosbestic). We'll also remove the artifacts we stored at the previous step and normalize the signal by dividing it with the isosbestic channel linear fit, obtaining our dF/F. 

4. Remove slow decay due to photobleaching, with high-pass filtering or detrending.

5. Low-pass filtering to remove noise.

Note that different groups do preprocessing differently and there is no universally accepted best practice for how to preprocess photometry data.  The best way to preprocess your data may depend on the details of the experimental setup and the questions you want to ask of the data. It is good practice to always visually inspect the raw data and the results of each preprocessing step to make sure they look sensible.  

The first step will be to clone the fiber photometry repository and to install requirements to the remote environment :

In [ ]:
# Clone repo
!git clone -b Develoment_Magendie https://github.com/AliceFermigier/Fiberphotometry_analysis.git
%cd Fiberphotometry_analysis

# Install dependencies
%pip install -r requirements.txt

We will then import the data that we need for the workshop. We're running our notebook on Colab, which means that our code is running on Google's cloud servers. Because of this we need to download the data we want to work on in our workspace. The following cell downloads the dataset from Drive:

In [ ]:
# Create the data folder if it doesn't exist
import os
os.makedirs("data", exist_ok=True)

# Download dataset
%pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/1F0OSujPWKy-UE_dgwpaYCdHSC7p1hGQ3 -O data

Import the standard python modules needed for the analysis.

In [ ]:
import numpy as  np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from dash import Dash, dcc, html, Input, Output, State
import plotly.express as px
import importlib
from scipy.signal import butter, filtfilt, detrend
from scipy.optimize import curve_fit
from ast import literal_eval

#set default plot properties
plt.rcParams['figure.figsize'] = [16, 12] # Make default figure size larger.
plt.rcParams['axes.xmargin'] = 0          # Make default margin on x axis zero.
plt.rcParams['axes.labelsize'] = 12     #Set default axes label size 
plt.rcParams['axes.titlesize']=15
plt.rcParams['axes.titleweight']='heavy'
plt.rcParams['ytick.labelsize']= 10
plt.rcParams['xtick.labelsize']= 10
plt.rcParams['legend.fontsize']=12
plt.rcParams['legend.markerscale']=2

Import the modules containing the helper functions. A lot of our functions are contained in dedicated files to make the file more readable (this is a common organization of pipelines).

In [ ]:
import modules.common.preprocess as pp
importlib.reload(pp)

import modules.common.genplot as gp
importlib.reload(gp)

import modules.common.nomenclature as nom
importlib.reload(nom)

We will also create a new "Preprocessing" directory to store our preprocessed data, inside our \data directory :

In [ ]:
data_path_exp = Path('/data')
pp_path = nom.setup_preprocessing_directory(data_path_exp)

# Load signals

We have just downloaded some fiberphotometry files to the folder \data. Our signals were recorded using Doric Neuroscience Studio, who use their own data storage format (.doric). The format resembles a HDF5 format, which hierarchically stores data and corresponding metadata. In order to access it we will use the load_deinterleaved_doric helper function, contained in the preprocess.py module. Let's try to load the "mouse1.doric".

In [ ]:
mouse = 'mouse1'
exp = 'workshop'

raw_data_path = f'{data_path_exp}/{mouse}.doric'
deinterleaved_df = pp.load_deinterleaved_doric(raw_data_path)
deinterleaved_df.to_csv(pp_path / f'{mouse}_deinterleaved.csv', index=False)

# Raw signals

Let's take a look at the raw GCamP signals. The blue curve corresponds to what the sensor emitted in response to blue light (465nm) stimulation and the purple curve (405nm) to the purple light stimulation. We will also automatically save the figure.

In [ ]:
fig_raw = gp.plot_rawdata(deinterleaved_df, exp, mouse)
fig_raw.savefig(pp_path / f'{mouse}_rawdata.png')

We can also plot this with the plotly library, which enables interactive plotting with Jupyter notebook.

In [ ]:
gp.plot_rawdata_interactive(deinterleaved_df, exp, mouse)

 As 405nm is the isosbestic point of the sensor, the emitted signal at 405nm is independent of calcium concentration, meaning that all the movemements observed in this curve correspond to movement of hemodynamic artifacts. What can you deduce from these curves?

# 1 - Remove high electrical artifacts

Now load and observe the signal stored in the file mouse2.doric. In the 405nm signal, high electrical noise can be observed between 490 and 500 seconds of recording. This kind of artifacts can be observed regularly in fiber photometry data, and does not correspond to physiological signal. 
To try and remove these artifacts, we will do a first cleaning of our signals with a Hampel filter. The function also plots the results to assess the efficiency of the filtering.

In [ ]:
def hampel_filter(data, window_size, n_sigmas=5):
    k = 1.4826  # scaling factor for Gaussian distribution
    
    is_series = isinstance(data, pd.Series)
    original_data = data.values if is_series else data
    new_data = original_data.copy()

    artifact_idx = []

    for i in range(window_size, len(original_data) - window_size):
        window = original_data[i - window_size:i + window_size + 1]
        median = np.nanmedian(window)
        mad = k * np.nanmedian(np.abs(window - median))

        if np.abs(original_data[i] - median) > n_sigmas * mad:
            new_data[i] = median
            artifact_idx.append(i)

    filtered = pd.Series(new_data, index=data.index) if is_series else new_data

    return filtered, artifact_idx

def plot_hampel_results(time, raw_405, raw_465, filt_405, filt_465, art405, art465):

    df_plot = pd.DataFrame({
        "Time": time,
        "Raw 405": raw_405,
        "Filtered 405": filt_405,
        "Raw 465": raw_465,
        "Filtered 465": filt_465
    })

    df_long = df_plot.melt(id_vars="Time", var_name="Signal", value_name="Value")

    fig = px.line(
        df_long,
        x="Time",
        y="Value",
        color="Signal",
        title="Raw vs Hampel Filtered Signals"
    )

    # Artifact markers
    fig.add_scatter(
        x=time.iloc[art405],
        y=filt_405.iloc[art405],
        mode="markers",
        marker=dict(size=8),
        name="Artifacts 405",
        hovertemplate="Artifact<br>Time: %{x}<br>Value: %{y}"
    )

    fig.add_scatter(
        x=time.iloc[art465],
        y=filt_465.iloc[art465],
        mode="markers",
        marker=dict(size=8),
        name="Artifacts 465",
        hovertemplate="Artifact<br>Time: %{x}<br>Value: %{y}"
    )

    fig.show()

def remove_high_artifacts(rawdata_df):

    time = rawdata_df['Time(s)']
    deinterleaved_405 = rawdata_df['405 Deinterleaved']
    deinterleaved_465 = rawdata_df['465 Deinterleaved']

    hampel_405, artifacts_405 = hampel_filter(deinterleaved_405, window_size=5, n_sigmas=5)
    hampel_465, artifacts_465 = hampel_filter(deinterleaved_465, window_size=5, n_sigmas=5)

    print(f"405 artifacts removed: {len(artifacts_405)}")
    print(f"465 artifacts removed: {len(artifacts_465)}")

    plot_hampel_results(
        time,
        deinterleaved_405,
        deinterleaved_465,
        hampel_405,
        hampel_465,
        artifacts_405,
        artifacts_465
    )

    hampel_df = pd.DataFrame({
        'Time(s)': time,
        '405 Deinterleaved': hampel_405,
        '465 Deinterleaved': hampel_465
        })
    
    return hampel_df

# Apply the function on your data
hampel_df = remove_high_artifacts(deinterleaved_df)

What can you deduce from the plots ? What parameters can you change to modify the outcome of the hampel filter ?

# 2 - Manual artifact scoring (if needed)

Now open the mouse3.doric file, and do a first cleaning with the hampel filter following the same steps we just followed. Can you see the large artifacts where the signal drops and reincreases abrubtly? This is caused by disconnections of the mouse's head implant from the patch cord, and it prevents good fitting of the isosbestic curve on the signal. Try to jump directly to step 3 to see what problems these artifacts can generate.

To limitate distortion of the dF/F, we can manually score very high artifacts and take them into account during the isosbestic fitting.

To do so, we'll open ou isosbestic channel and click on the beginning and then the end of each of the disconnection artifacts. The timestamps will be stored in a separate excel, called "artifacts.xlsx". We will create it with a helper function :

In [ ]:
artifact_file = f'{data_path_exp}/artifacts.xlsx' # File to store artifact timestamps
nom.create_or_load_artifacts_file(artifact_file, option='create_only')

We will then open our isosbestic curve in an interactive window, and score our artifacts :

In [ ]:
# The artifacts will be stored with the filecode associated with the name of the mouse and the experiment
filecode = f'{exp}_{mouse}'

# Create the Dash app
app = Dash(__name__)

# Create the figure
fig = px.line(deinterleaved_df, x='Time(s)', y='405 Deinterleaved')

# App layout
app.layout = html.Div([
    html.H4(f'{exp} {mouse}'),
    
    dcc.Graph(
        id='plot',
        figure=fig,
        config={'displayModeBar': True}  # Add buttons for zooming, panning, etc.
    ),
    
    html.Div(id='artifact-message', style={'color': 'black', 'fontWeight': 'bold'}),
    
    html.Button("Save Artifacts", id="save-button", n_clicks=0),
    
    dcc.Store(id='artifact-storage', data=[]),  # Store artifact tuples (start, end)
    dcc.Store(id='click-tracker', data=None)  # Keep track of first/second click
])

# Callback to handle user clicks and record artifact intervals
@app.callback(
    [Output('artifact-storage', 'data'),
     Output('artifact-message', 'children'),
     Output('click-tracker', 'data')],
    Input('plot', 'clickData'),
    [State('artifact-storage', 'data'),
     State('click-tracker', 'data')]
)
def capture_artifact(click_data, artifact_intervals, click_state):
    """
    Handles clicks on the plot. 
    On the first click, the start of the artifact is captured. 
    On the second click, the end of the artifact is captured, and the interval is saved.
    """
    if click_data:
        time_clicked = click_data['points'][0]['x']
        
        if click_state is None:  # First click (start of the artifact)
            click_state = time_clicked
            message = f'Artifact start marked at {time_clicked:.2f} seconds. Now click the end point.'
            print(message)
        else:  # Second click (end of the artifact)
            start = min(click_state, time_clicked)
            end = max(click_state, time_clicked)
            artifact_intervals.append((start, end))
            message = f'Artifact interval ({start:.2f}s, {end:.2f}s) saved. Click to start a new interval.'
            print(message)
            click_state = None  # Reset click state for next pair of clicks

    else:
        message = 'Click on the graph to mark the start of an artifact.'

    return artifact_intervals, message, click_state

# Callback to save artifact intervals to an Excel file
@app.callback(
    Output('save-button', 'children'),
    Input('save-button', 'n_clicks'),
    State('artifact-storage', 'data')
)
def save_artifacts_to_excel(n_clicks, artifact_intervals):
    """
    Saves the artifact intervals to an Excel file when the save button is pressed.
    Each row in the Excel file contains the start and end times of each artifact.
    """
    if n_clicks > 0:
        if len(artifact_intervals) > 0:
            print(f"\n--- Processing filecode: {filecode} ---")
            print(f"Artifacts to store: {artifact_intervals}")
            pp.update_artifacts_file(artifact_file, filecode, artifact_intervals)
            print(f"Saved {len(artifact_intervals)} artifact intervals to {artifact_file}")
            return f'Saved {len(artifact_intervals)} Artifacts'
        else:
            print("No artifacts to save.")
            return "No artifacts to save"

    return "Save Artifacts"

# Run the server
if __name__ == '__main__':
    app.run(debug=False, use_reloader=False)

After finishing scoring your artifacts, go check the artifacts excel. Can you see how your artifacts are stored ?
We will use these timestamps in the following step to better our movement correction.

# 3 - Removing movement artifacts with the isosbestic signal

When you excite GCaMP with a LED at 405nm, the light emitted by GCaMP is independent of the calcium release. This is very useful to differentiate if the observed variations in our signal are due to real neuronal activity or to movement and/or hemodynamic artifacts. Basically, if a variation is mirrorred in the isosbestic curve, the variation is an artifact. If not, it is real neuronal activity.

To remove motion artifacts, we want first to match our isosbestic curve (405 nm) to our calcium-dependent signal (465 nm). Two options are available :
- 'mean' : normalize both 405 and 465 independently, with their own mean
- 'fit' : fit the 405 onto the 465nm curve (recommended)

We will then substract compute our dFF, with the isosbestic curve as baseline :
$$
dF/F = \frac{465 - \text{fitted405}}{\text{fitted405}} \times 100
$$

In [ ]:
# Helper function for the linear fitting

def linearfit_sklearn(sig_405, sig_465):
    # Fit 405 to 465 using linear regression
    from sklearn.linear_model import LinearRegression
    model = LinearRegression()
    
    if isinstance(sig_405, np.ndarray):
        sig_405 = sig_405.reshape(-1, 1)
    else:
        sig_405 = sig_405.to_numpy().reshape(-1, 1)
        sig_465 = sig_465.to_numpy()

    model.fit(sig_405, sig_465)
    fitted_405 = model.predict(sig_405)

    return fitted_405

Included in these functions are also the tools to remove artifacts. Indeed, when big disconnections happen, the fitting is not ideal when computed on the whole signal. To better the fitting, we will slice our signal where the disconnections happenned and treat our signal bit by bit.

In [ ]:
def remove_artifacts(data_df, artifact_intervals, col, method='fit'):
    """
    Helper function to remove artifacts from a specific column of the data.
    
    Parameters:
    - data_df (pd.DataFrame): Input data
    - artifact_intervals (list of tuples): List of artifact intervals as [(start, stop), ...]
    - col (str): Column to process ('405 Deinterleaved' or '465 Deinterleaved')
    - begin (int): Starting index for the segment
    - end (int): Ending index for the segment
    - sr (int): Sampling rate
    - method (str): 'mean' or 'fit' method to compute dFF
    
    Returns:
    - Tuple: Updated dFF segment, updated 'begin' index, and 'end' index
    """
    begin=0
    dFF_segment = np.full(len(data_df), np.nan)  # Create an array filled with NaNs of the same length as data_df
    artifact_intervals.append([len(data_df), 'End'])
    
    for x_start, x_stop in artifact_intervals:
        try:
            # Calculate 'end' as the first index where 'Time(s)' is greater than x_start, -1 to not overlap with artifact
            end = data_df.index[data_df['Time(s)'] < x_start][-1]
            
            # Extract the segment of data for processing
            segment = data_df.iloc[begin+1:end][col].values  # Use iloc for absolute indexing
            
            if method == 'mean':
                mean_fluorescence = np.nanmean(segment)
                dFF_values = ((segment - mean_fluorescence) / mean_fluorescence) * 100
                # Check for length match before assignment
                if len(dFF_values) == len(dFF_segment[begin+1:end]):
                    dFF_segment[begin+1:end] = dFF_values
                else:
                    print(f"Shape mismatch: dFF_values ({len(dFF_values)}) vs dFF_segment ({len(dFF_segment[begin+1:end])})")
            
            elif method == 'fit':
                dFF_values = linearfit_sklearn(data_df.iloc[begin+1:end]['405 Deinterleaved'].values, 
                                            data_df.iloc[begin+1:end]['465 Deinterleaved'].values)
                if len(dFF_values) == len(dFF_segment[begin+1:end]):
                    dFF_segment[begin+1:end] = dFF_values
                else:
                    print(f"Shape mismatch: dFF_values ({len(dFF_values)}) vs dFF_segment ({len(dFF_segment[begin+1:end])})")
            
            # Update 'begin' to the index just before the stop of the artifact
            if x_stop != 'End':
                begin = data_df.index[data_df['Time(s)'] > x_stop][0]
                
        except Exception as e:
            print(f"Error processing artifact interval ({x_start}, {x_stop}): {e}")
    
    return dFF_segment

We then compute the dFF on our signal

In [ ]:
def dFF(data_df, artifacts_df, filecode, method='fit'):
    """
    Calculates dFF (delta F over F) and removes artifacts from 405nm and 465nm photometry data.
    
    Parameters:
    - data_df (pd.DataFrame): Input photometry data containing 'Time(s)', '405 Deinterleaved', '465 Deinterleaved'
    - artifacts_df (pd.DataFrame): Dataframe containing artifact information
    - filecode (str): Unique identifier for the file being processed
    - sr (int): Sampling rate of the data
    - method (str): 'mean' or 'fit' method for calculating dFF
    
    Returns:
    - dFFdata_df (pd.DataFrame): DataFrame with 'Time(s)', '405 dFF', '465 dFF', and 'Denoised dFF'
    """
    dFFdata = np.full([3, len(data_df)], np.nan)

    if method == 'mean':
        for i, col in enumerate(['405 Deinterleaved', '465 Deinterleaved']):
            if filecode in artifacts_df['Filecode'].values:
                artifact_intervals = artifacts_df.loc[artifacts_df['Filecode'] == filecode, 'Artifacts'].values
                artifact_intervals = literal_eval(artifact_intervals[0])
                dFFdata[i] = remove_artifacts(data_df, artifact_intervals, col, method='mean')
            else:
                mean_fluorescence = np.nanmean(data_df[col])
                dFFdata[i] = ((data_df[col] - mean_fluorescence) / mean_fluorescence) * 100
    
    elif method == 'fit':
        if filecode in artifacts_df['Filecode'].values:
            artifact_intervals = artifacts_df.loc[artifacts_df['Filecode'] == filecode, 'Artifacts'].values
            artifact_intervals = literal_eval(artifact_intervals[0])
            dFFdata[0] = remove_artifacts(data_df, artifact_intervals, '405 Deinterleaved', method='fit')
            dFFdata[1] = data_df['465 Deinterleaved'].to_numpy()
        else:
            dFFdata[0] = linearfit_sklearn(data_df['405 Deinterleaved'], data_df['465 Deinterleaved'])
            dFFdata[1] = data_df['465 Deinterleaved'].to_numpy()

    # Calculate Denoised dFF
    dFFdata[2] = ((dFFdata[1] - dFFdata[0]) / dFFdata[0]) * 100

    dFFdata_df = pd.DataFrame({
        'Time(s)': data_df['Time(s)'],
        '405 dFF': dFFdata[0],
        '465 dFF': dFFdata[1],
        'Denoised dFF': dFFdata[2]
    })

    return dFFdata_df

Lastly, when artifacts are removed, NaNs appear in the signal. We want to fill these missing values, otherwise it will cause bugs later in our pipeline.
We will interpolate data, and have a few different options :
- mean pad : replaces missing values with the column mean. Good for very noisy datasets.
- linear : draws a straight line between surrounding points. Good for short artifact periods.
- cubic spline : produces a smooth curve instead of straight segments. Often recommended because patch is smoother, but can invent unrealistic peaks if artifact is too long or signal is too unstable.

In [ ]:
def interpolate_dFFdata(data_df, method='linear'):
    """
    Interpolates or fills NaN values in dFF data.
    
    Parameters:
    -----------
    data_df : pd.DataFrame
        DataFrame containing columns for '405 dFF', '470 dFF', 'Denoised dFF' with NaN values if artifacts were removed
        
    method : str, optional
        The method to fill NaN values. Options:
        - 'linear' : Linearly interpolates NaN values between valid data points.
        - 'mean pad' : Replaces all NaN values with the mean of the respective column.
        
    Returns:
    --------
    data_df : pd.DataFrame
        The same DataFrame, but with NaN values filled.
    """
    
    if method not in ['linear', 'mean pad']:
        raise ValueError(f"Unsupported method: '{method}'. Choose 'linear','mean pad' or 'cubic spline'")
    
    # Process only the dFF columns (ignoring 'Time(s)')
    dff_columns = data_df.columns[1:]  # Exclude 'Time(s)' column
    
    if method == 'linear':
        # Use interpolate with 'both' to ensure it fills NaNs at the beginning and end
        data_df[dff_columns] = data_df[dff_columns].interpolate(method='linear', limit_direction='both')
    
    elif method == 'mean pad':
        # Compute the mean of each column and fill NaNs with this mean
        col_means = data_df[dff_columns].mean(skipna=True)  # Mean of each dFF column, ignoring NaNs
        data_df[dff_columns] = data_df[dff_columns].fillna(col_means)

    elif method == 'cubic spline':
        data_df[dff_columns] = data_df[dff_columns].interpolate(
            method='spline',
            order=3
        )
        
    return data_df

Let's apply all these on our data !

In [ ]:
#import artifacts boundaries
artifacts_df = pd.read_excel(artifact_file)

#define the method you want to use to match the isosbestic to your calcium dependent signal
method = 'fit'

# calculate dFF with artifacts removal, then interpolate missing data
dFFdata_df = pp.dFF(hampel_df,artifacts_df,filecode,method)
interpdFFdata_df = pp.interpolate_dFFdata(dFFdata_df, method='linear')

#sometimes 1st timestamps=Nan instead of 0, raises an error
interpdFFdata_df['Time(s)'] = interpdFFdata_df['Time(s)'].fillna(0) 

#plotted GCaMP and isosbestic curves after dFF
fig_dFF = gp.plot_fiberpho(interpdFFdata_df,exp,mouse,method)
fig_dFF.savefig(pp_path/f'{mouse}_{method}_dFF.png')

# 4 - Remove slow decay due to photobleaching

The longer you expose your GCaMP to light, the less the fluorophore will emit light. This phenomenon is called photobleaching and causes the signal to decrease slowly with time. Did you observe the decay in your raw signal?

Several methods are available to correct for this decay in our signal. We will first try with a high-pass filter, that removes very slow oscillations. We will high pass at 0.01Hz, which correponds to a period of 1,6 minutes.

### 4.1 - High-pass filter

In [ ]:
def highpass_filter(data_df, sr, cutoff=0.01, order=1):
    """
    High-pass filters the signal to remove slow trends.

    Parameters:
    - signal: 1D numpy array or list of your raw fluorescence values
    - cutoff: cutoff frequency in Hz (e.g., 0.01 Hz = 100 sec cycles)
    - fs: sampling rate in Hz (10 Hz in your case)
    - order: filter order (higher = sharper cutoff)

    Returns:
    - detrended signal as a NumPy array
    """
    nyq = 0.5 * sr
    norm_cutoff = cutoff / nyq
    b, a = butter(order, norm_cutoff, btype='high', analog=False)
    filtered_signal = filtfilt(b, a, data_df)
    return filtered_signal

def highpass_filter_with_padding(signal, sr, cutoff=0.01, order=1, pad_seconds=50):
    pad_len = int(sr * pad_seconds)
    pre_pad = signal[:pad_len][::-1] if pad_len < len(signal) else signal[::-1]
    post_pad = signal[-pad_len:][::-1] if pad_len < len(signal) else signal[::-1]
    padded = np.concatenate([pre_pad, signal, post_pad])

    nyq = 0.5 * sr
    norm_cutoff = cutoff / nyq
    b, a = butter(order, norm_cutoff, btype='high', analog=False)
    filtered = filtfilt(b, a, padded)

    return filtered[pad_len:-pad_len]

def highpass_filter_dff(dff, padding = False):
    sr = pp.samplerate(dff)
    cutoff_freq = 0.01
    denoised_dff = dff['Denoised dFF']
    time = dff['Time(s)']

    if padding :
        filtered_denoised_dff = highpass_filter_with_padding(
            denoised_dff, sr, cutoff=cutoff_freq, order=1, pad_seconds=50
        )
    else :
        filtered_denoised_dff = highpass_filter(
            denoised_dff, sr, cutoff=cutoff_freq, order=1
        )

    # Plot settings
    fig, axs = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw={'height_ratios': [1, 1]})
    
    # Unfiltered
    axs[0].plot(time, denoised_dff, color='black', linewidth=1)
    axs[0].set_title('Unfiltered dF/F')
    axs[0].set_ylabel('dF/F (%)')

    # Filtered
    axs[1].plot(time, filtered_denoised_dff, color='seagreen', linewidth=1)
    axs[1].set_title(f'Filtered dF/F (High-pass {cutoff_freq} Hz) - Padding : {padding}')
    axs[1].set_xlabel('Time (s)')
    axs[1].set_ylabel('dF/F (%)')

    # Adjust layout
    plt.tight_layout()
    plt.show()

    dff['Denoised dFF'] = filtered_denoised_dff
    return dff

Let's apply our function to our data :

In [ ]:
#high-pass filter to remove slow oscillations
filtered_dFFdata = highpass_filter_dff(interpdFFdata_df, padding = False)

Can you see that the filtering is not optimal at the beginning and ending of our trace ? This is a known edge effect that happens when filtering data. To prevent this from happening, we will do a padding, meaning that we will add a certain time at the beginning and end of our data by mirroring our curve, applying the filter on the new trace, and then removing the data artificially created at the beginning and ending.

In [ ]:
#high-pass filter to remove slow oscillations, with padding
filtered_dFFdata = highpass_filter_dff(interpdFFdata_df, padding = True)
filtered_dFFdata.to_csv(pp_path/f'{mouse}_highpass_dFF.csv')

### 4.2 - Detrending

Another option is to use a detrend function. The detrend function from the SciPy module scipy.signal removes systematic trends from a signal so that the remaining data fluctuate around zero. It does this by estimating a trend (either a constant mean or a linear least-squares fit) and subtracting it from the data. 

Let's see how this option performs.

In [ ]:
def detrend_dff(dff, type='linear'):
    
    denoised_dff = dff['Denoised dFF']
    time = dff['Time(s)']
    detrended_dff = detrend(denoised_dff, type='linear')

    # Plot settings
    fig, axs = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw={'height_ratios': [1, 1]})

    # Unfiltered
    axs[0].plot(time, denoised_dff, color='black', linewidth=1)
    axs[0].set_title('Unfiltered dF/F')
    axs[0].set_ylabel('dF/F (%)')

    # Filtered
    axs[1].plot(time, detrended_dff, color='seagreen', linewidth=1)
    axs[1].set_title(f'Detrended dF/F - Type : {type}')
    axs[1].set_xlabel('Time (s)')
    axs[1].set_ylabel('dF/F (%)')

    # Adjust layout
    plt.tight_layout()
    plt.show()

    dff['Denoised dFF'] = detrended_dff
    return dff

Let's apply this function to our data :

In [ ]:
detrended_dFFdata = detrend_dff(interpdFFdata_df, type='linear')
detrended_dFFdata.to_csv(pp_path/f'{mouse}_detrend_linear_dFF.csv')

When type='linear', the function fits a straight line to the signal using least-squares regression and subtracts this line from the original data, effectively removing slow baseline drift while preserving higher-frequency variations in the signal. However, this assumes that our drift is linear whereas in many fiber photometry pipelines, bleaching is modeled as an exponential decay, because fluorescence loss over time often follows this shape more closely than a straight line. The typical approach is to fit an exponential curve to the baseline drift and subtract (or divide) it from the signal.

In [ ]:
def exp_func(t, A, tau, C):
    return A * np.exp(-t / tau) + C

def exponential_detrend(dff):

    denoised_dff = dff['Denoised dFF']
    time = dff['Time(s)']

    # Initial parameter guesses
    p0 = [np.max(denoised_dff), np.mean(time), np.min(denoised_dff)]

    # Fit exponential decay
    params, _ = curve_fit(exp_func, time, denoised_dff, p0=p0)

    trend = exp_func(time, *params)

    detrended_dff = denoised_dff - trend

    # Plot settings
    fig, axs = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw={'height_ratios': [1, 1]})

    # Unfiltered
    axs[0].plot(time, denoised_dff, color='black', linewidth=1)
    axs[0].set_title('Unfiltered dF/F')
    axs[0].set_ylabel('dF/F (%)')

    # Filtered
    axs[1].plot(time, detrended_dff, color='seagreen', linewidth=1)
    axs[1].set_title(f'Detrended dF/F - Type : exponential')
    axs[1].set_xlabel('Time (s)')
    axs[1].set_ylabel('dF/F (%)')

    # Adjust layout
    plt.tight_layout()
    plt.show()

    dff['Denoised dFF'] = detrended_dff
    return dff

In [ ]:
detrended_dFFdata = exponential_detrend(interpdFFdata_df)
detrended_dFFdata.to_csv(pp_path/f'{mouse}_detrend_exponential_dFF.csv')

# 5 - Low-pass filter to remove high-frequency noise

Lastly, depending on the quality of our signal and on our needs, we can remove high frequency noise to improve the look of our curves and better isolate the meaningful variations.

We will do this with a lowpass zero phase filter, starting with a 10Hz cutoff frequency.

In [ ]:
def lowpass_dFF(dff, order = 2, cut_freq = 10):

    sampling_rate = pp.samplerate(dFF)
    time = dff['Time(s)']
    raw_dff = dff['Denoised dFF']

    # Lowpass filter - zero phase filtering (with filtfilt) is used to avoid distorting the signal.
    b,a = butter(order, cut_freq, btype='low', fs=sampling_rate)
    dFF_lowpass = filtfilt(b,a, raw_dff)

    fig,ax1=plt.subplots()
    ax1.plot(time, raw_dff, 'g', alpha=0.3, label='dFF raw')
    ax1.plot(time, dFF_lowpass, 'g', label='dFF lowpass')

    ax1.set_xlabel('Time(s)')
    ax1.set_ylabel('dF/F (%)', color='g')
    ax1.set_title('Denoised signal')

    dff['Denoised lowpass dFF'] = dFF_lowpass
    return dff

Let's apply this to our signal :

In [ ]:

denoised_dFFdata = lowpass_dFF(detrended_dFFdata, order = 2, cut_freq = 10)

We can again plot this in an interactive window to see how the filtering altered our signal :

In [ ]:
# Create the interactive plot
fig = gp.plot_denoised_photometry(denoised_dFFdata)

Try these with different cutting frequencies to see how it alters your signal. Is there a cutting frequency that seems optimal to you?